In [ ]:
# ── Cell 1: Imports & Parameters ─────────────────────────────────────────────
# Notebook 12g: Random Forest — Real Only Condition (EuroSAT)
# Train on merged real EuroSAT train+val
# Test on real EuroSAT test set
# Hyperparameters fixed from notebook 12a HP search

import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

SEED = 42
np.random.seed(SEED)

IMAGE_SIZE = 64
CHANNELS   = 3
CLASSES    = 10
CLASS_NAMES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]

# Fixed hyperparameters from notebook 12a HP search
CRITERION    = 'entropy'
MAX_DEPTH    = 100
N_ESTIMATORS = 100

PROJECT_DIR = '/content/drive/MyDrive/thesis_project'  # adjust to your Drive path
REAL_PATH   = os.path.join(PROJECT_DIR, 'data', 'processed', 'seed_42', 'eurosat.npz')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results', 'rf', 'eurosat', 'real_only')
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 2: Data Loading ──────────────────────────────────────────────────────

real_data    = np.load(REAL_PATH)
X_train_real = real_data['X_train']
y_train_real = real_data['y_train']
X_val_real   = real_data['X_val']
y_val_real   = real_data['y_val']
X_test       = real_data['X_test']
y_test       = real_data['y_test']

# Merge real train + val
X_train = np.concatenate([X_train_real, X_val_real], axis=0)
y_train = np.concatenate([y_train_real, y_val_real], axis=0)

print(f'Real train: {X_train_real.shape}, val: {X_val_real.shape}')
print(f'Merged real train: {X_train.shape}')
print(f'Real test:  {X_test.shape}')

In [ ]:
# ── Cell 3: Preprocessing ─────────────────────────────────────────────────────

def preprocess(X):
    return X.astype('float32').reshape(len(X), -1) / 255.0

X_train_n = preprocess(X_train)
X_test_n  = preprocess(X_test)

In [ ]:
# ── Cell 4: Train & Evaluate ──────────────────────────────────────────────────

clf = RandomForestClassifier(
    criterion=CRITERION,
    max_depth=MAX_DEPTH,
    n_estimators=N_ESTIMATORS,
    random_state=SEED,
    n_jobs=-1
)
clf.fit(X_train_n, y_train)

y_pred   = clf.predict(X_test_n)
test_acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {test_acc:.4f}')

report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4)
print('\nClassification Report:')
print(report)

report_dict = classification_report(
    y_test, y_pred, target_names=CLASS_NAMES, digits=4, output_dict=True
)
pd.DataFrame(report_dict).transpose().to_csv(
    os.path.join(RESULTS_DIR, 'classification_report.csv')
)
pd.DataFrame([{
    'condition': 'real_only',
    'dataset':   'eurosat',
    'test_acc':  round(test_acc, 4),
}]).to_csv(os.path.join(RESULTS_DIR, 'summary.csv'), index=False)

print(f'\nResults saved to: {RESULTS_DIR}')